# 02 — Exploratory Data Analysis (EDA)

**Ziel**: Datenverteilungen verstehen, Plausibilität prüfen, Findings für die Dokumentation festhalten.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
ABSTRACTS_PATH = Path('../data/pubmed_abstracts')
DATA_PROCESSED.mkdir(exist_ok=True)

cycle_df = pd.read_csv(DATA_RAW / 'cycle_tracking.csv')
workout_df = pd.read_csv(DATA_RAW / 'workouts.csv')

with open(ABSTRACTS_PATH / 'pubmed_abstracts.jsonl', 'r') as f:
    abstracts = [json.loads(line) for line in f]

print(f'Cycle: {cycle_df.shape} | Workouts: {workout_df.shape} | Abstracts: {len(abstracts)}')

## 1. Cycle-Tracking-Daten

In [ ]:
cycle_df.describe()

In [ ]:
# Phasen-Verteilung
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
cycle_df['phase'].value_counts().plot(kind='bar', ax=axes[0], color='#c08fb5')
axes[0].set_title('Phasen-Verteilung')
axes[0].set_ylabel('Anzahl Tage')

cycle_df['recommended_intensity'].value_counts().plot(kind='bar', ax=axes[1], color='#8fb5c0')
axes[1].set_title('Empfehlungs-Label-Verteilung')
axes[1].set_ylabel('Anzahl')
plt.tight_layout()
plt.savefig('../docs/screenshots/eda_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# BBT über Zyklustage
plt.figure(figsize=(12, 5))
for phase in ['menstruation', 'follicular', 'ovulation', 'luteal']:
    subset = cycle_df[cycle_df['phase'] == phase]
    plt.scatter(subset['day_in_cycle'], subset['bbt_celsius'], alpha=0.1, label=phase, s=10)
plt.xlabel('Tag im Zyklus')
plt.ylabel('Basal Body Temperature (°C)')
plt.title('BBT-Verlauf über den Zyklus (biphasisches Muster)')
plt.legend()
plt.savefig('../docs/screenshots/eda_bbt_curve.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Schlafqualität je Phase
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
phase_order = ['menstruation', 'follicular', 'ovulation', 'luteal']
sns.boxplot(data=cycle_df, x='phase', y='sleep_quality', order=phase_order, ax=axes[0], palette='pastel')
axes[0].set_title('Schlafqualität je Phase')
sns.boxplot(data=cycle_df, x='phase', y='sleep_hours', order=phase_order, ax=axes[1], palette='pastel')
axes[1].set_title('Schlafdauer je Phase')
plt.tight_layout()
plt.savefig('../docs/screenshots/eda_sleep_per_phase.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Symptom-Häufigkeit je Phase
from collections import Counter

symptom_counts = {p: Counter() for p in phase_order}
for _, row in cycle_df.iterrows():
    for s in row['symptoms'].split(';'):
        if s != 'none':
            symptom_counts[row['phase']][s] += 1

symptom_df = pd.DataFrame(symptom_counts).fillna(0).astype(int)
plt.figure(figsize=(10, 5))
sns.heatmap(symptom_df, annot=True, fmt='d', cmap='Reds')
plt.title('Symptom-Häufigkeit je Phase')
plt.savefig('../docs/screenshots/eda_symptoms_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

## 2. Workout-Daten

In [ ]:
workout_df.describe()

In [ ]:
# Recovery-Zeit je Phase und Intensität
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=workout_df, x='phase', y='recovery_hours', order=phase_order, ax=axes[0], palette='pastel')
axes[0].set_title('Recovery-Stunden je Phase')
sns.boxplot(data=workout_df, x='phase', y='rpe', order=phase_order, ax=axes[1], palette='pastel')
axes[1].set_title('RPE (gefühlte Anstrengung) je Phase')
plt.tight_layout()
plt.savefig('../docs/screenshots/eda_workout_phases.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. PubMed-Abstracts

In [ ]:
abs_df = pd.DataFrame(abstracts)
abs_df['abstract_length'] = abs_df['abstract'].str.len()
print(f'Anzahl Abstracts: {len(abs_df)}')
print(f'Durchschnittliche Länge: {abs_df["abstract_length"].mean():.0f} Zeichen')
print(f'Median Jahr: {abs_df["year"].median()}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
abs_df['year'].dropna().astype(int).hist(bins=30, ax=axes[0], color='#b58fc0')
axes[0].set_title('Publikationsjahre der Abstracts')
axes[0].set_xlabel('Jahr')
abs_df['abstract_length'].hist(bins=30, ax=axes[1], color='#8fc0b5')
axes[1].set_title('Abstract-Länge (Zeichen)')
plt.tight_layout()
plt.savefig('../docs/screenshots/eda_pubmed.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Processed Data Speichern

In [ ]:
# Cycle-Daten für Modell vorbereiten
# Symptome als binäre Features
symptom_list = ['cramps', 'fatigue', 'mood_low', 'headache', 'bloating', 'tender_breasts']
for s in symptom_list:
    cycle_df[f'sym_{s}'] = cycle_df['symptoms'].apply(lambda x: int(s in x.split(';')))

cycle_df['symptom_count'] = cycle_df[[f'sym_{s}' for s in symptom_list]].sum(axis=1)

cycle_df.to_csv(DATA_PROCESSED / 'cycle_processed.csv', index=False)
print('Saved processed cycle data')
cycle_df.head()

## Key Findings (für Doku)

- **BBT folgt erwartetem biphasischen Muster** (Anstieg um Ovulation, Plateau in Luteal)
- **Symptomlast** korreliert stark mit Phase (Menstruation + späte Luteal = höchste Last)
- **Schlafqualität** in Lutealphase und Menstruation niedriger
- **Label-Verteilung** für Trainingsempfehlung leicht imbalanced → bei Modelltraining beachten
- **PubMed-Abstracts** decken Spannweite ab 1980er bis 2024, ausreichend für RAG